In [ ]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import datetime

# Making a GET request
r = requests.get('https://live-drillingcontractor-org.pantheonsite.io/news')# 'https://www.geeksforgeeks.org/python-programming-language/')

# Parsing the HTML
soup = BeautifulSoup(r.content, 'html.parser')
# Look into post list
s = soup.find('div', class_='mag-box-container clearfix')
s1 = soup.find_all('li', class_='post-item tie-standard') # find post items
# content = s1.find_all('a')

# print(content)
# print(s)

In [216]:
# From post list, get titles and links
post_titles = [si.a['aria-label'] for si in s1]
post_links = [si.a['href'] for si in s1]
post_dates = [si.span.text.strip() for si in s1]

### Filter which articles contain the keywords

In [224]:
# Filter which articles contain the keywords
# "mpd", "managed pressure drilling", "cml", "controlled mud level", "pmcd", "mud cap drilling"
mpd_bool = np.zeros(len(post_links))#[None] * len(post_titles)
for i in range(len(post_titles)):
  # Making a GET request
  r = requests.get(post_links[i])
  # Parsing the HTML
  soup = BeautifulSoup(r.content, 'html.parser')
  # Look into div entry-content
  s = soup.find('div', class_='entry-content')
  # Get main post text
  text_list = s.find_all('p')
  # print(s)
  # Check if MPD is within the text
  matches = ["mpd", "managed pressure drilling", "cml", "controlled mud level", "pmcd", "mud cap drilling"]
  for text_i in text_list:
    if any(x in str(text_i).lower() for x in matches):
      mpd_bool[i] = True
      break
    else:
      next

In [225]:
# Drilling Contractor posts dataframe
dc_posts = pd.DataFrame({'title': post_titles, 'link': post_links, 'date': post_dates, 'mpd_bool': mpd_bool})
# Filter articles containing MPD
dc_posts[dc_posts['mpd_bool'] != 0]

,title,link,date,mpd_bool


In [124]:
# Filter TRUE posts
[[post_titles[i], post_links[i]] for i in range(len(post_titles)) if mpd_bool[i] == True]

[]

### Search DC Archive

In [333]:
# Making a GET request - DC archive
r = requests.get('https://drillingcontractor.org/dc-archive')

# Parsing the HTML
soup = BeautifulSoup(r.content, 'html.parser')
# Look into post list
s1 = soup.find_all('td', width='75') # find post items
# Retrieve archive links
archive_links = []
# Get link for monthly edition (e.g. jan/feb 2025)
for si in s1:
  try:
    archive_links.append(si.a['href'])
  except:
    next

In [334]:
# Check inside archive
# Making a GET request
r = requests.get(archive_links[0])
# Parsing the HTML
soup = BeautifulSoup(r.content, 'html.parser')
s = soup.find('div', class_='entry-content entry clearfix')
s1 = s.find_all('tr')
# print(s1)
# Find links inside every edition link
archive_links_2 = []
for si in s1:
  try:
    archive_links_2.append(si.td.a['href'])
  except:
    next

In [342]:
def extract_date_dc_archive(soup):
    post_date_i = soup.find('span', class_='date meta-item tie-icon').text.strip()
    post_date_i = datetime.strptime(post_date_i, '%b %d, %Y')
    return post_date_i

In [343]:
# Making a GET request
r = requests.get(archive_links_2[1])
# Parsing the HTML
soup = BeautifulSoup(r.content, 'html.parser')

# Look into post text
s = soup.find('div', class_='entry-content entry clearfix') # find post items
s1 = s.find_all('p')
matches = ["mpd", "managed pressure drilling", "cml", "controlled mud level", "pmcd", "mud cap drilling"]
for text_i in s1:
  if any(x in str(text_i.text.strip()).lower() for x in matches):
    post_title_i = soup.find('h1', class_='post-title entry-title').text.strip() # find post items
    post_link_i = archive_links_2[1]
    post_date_i = extract_date_dc_archive(soup)
    # print(text_i.text.strip())

In [345]:
# Loop entire archive
post_title_arc_dc = []
post_link_arc_dc = []
post_date_archive_dc = []
for archive_link_i in archive_links:
  # Making a GET request
  r = requests.get(archive_link_i) # Get link for monthly edition (e.g. jan/feb 2025)
  # Parsing the HTML
  soup_i = BeautifulSoup(r.content, 'html.parser')
  s = soup_i.find('div', class_='entry-content entry clearfix')
  s1 = s.find_all('tr')
  # print(s1)
  # Find links inside every edition link
  archive_links_2 = []
  for si in s1:
    try:
      archive_links_2.append(si.td.a['href'])
    except:
      next
  for archive_link_ii in archive_links_2:
    # Making a GET request
    r = requests.get(archive_link_ii)
    # Parsing the HTML
    soup_ii = BeautifulSoup(r.content, 'html.parser')
    # Look into post text
    s = soup_ii.find('div', class_='entry-content entry clearfix') # find post items
    try:
      s1 = s.find_all('p')
      matches = ["mpd", "managed pressure drilling", "cml", "controlled mud level", "pmcd", "mud cap drilling"]
      for text_i in s1:
        if any(x in str(text_i.text.strip()).lower() for x in matches):
          post_title_i = soup_ii.find('h1', class_='post-title entry-title').text.strip() # find post items
          post_link_i = archive_link_ii
          post_date_i = extract_date_dc_archive(soup_ii)
          post_title_arc_dc.append(post_title_i)
          post_link_arc_dc.append(post_link_i)
          post_date_archive_dc.append(post_date_i)
          # print(text_i.text.strip())
    except:
      next

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [351]:
# Drilling Contractor Archive posts dataframe
dc_archive_posts = pd.DataFrame({'title': post_title_arc_dc, 'link': post_link_arc_dc, 'date': post_date_archive_dc})
dc_archive_posts.drop_duplicates(subset=['title'], inplace=True)

In [353]:
dc_archive_posts

,title,link,date
0,Value-driven innovations will stand out amid f...,http://www.drillingcontractor.org/?p=71525,2025-01-23
1,News Cuttings,http://www.drillingcontractor.org/?p=71486,2025-01-23
2,Noble drillship visit provides UT students wit...,http://www.drillingcontractor.org/?p=71459,2025-01-23
3,Drillers seek ways to maintain capital discipl...,http://www.drillingcontractor.org/?p=67663,2024-01-26
7,"People, Companies & Products",http://www.drillingcontractor.org/?p=67707,2024-01-26
12,"Projects in GOM, Mediterranean showcase expand...",http://www.drillingcontractor.org/?p=68253,2024-03-11
40,Drilling & Completion News,http://www.drillingcontractor.org/?p=68198,2024-03-11
44,Drilling & Completion Tech Digest,http://www.drillingcontractor.org/?p=68691,2024-04-24
47,Automation initiatives show untapped potential...,http://www.drillingcontractor.org/?p=69603,2024-07-11
50,"The IADC team: Honoring our past, celebrating...",http://www.drillingcontractor.org/?p=70513,2024-10-22


## Search Offshore Engineer

In [199]:
# Making a GET request
r = requests.get('https://www.oedigital.com/managed%20pressure%20drilling')# 'https://www.geeksforgeeks.org/python-programming-language/')

# Parsing the HTML
soup = BeautifulSoup(r.content, 'html.parser')
# Look into post list
s = soup.find('div', class_='article')
s1 = soup.find_all('a', class_='snippet')
# s1 = soup.find_all('li', class_='post-item tie-standard') # find post items
# print(s1)

In [186]:
def extract_post_date(s1):
  from datetime import datetime
  date=datetime.strptime(s1.div.text.strip(), '%b %d, %Y')
  return date

post_links = ['https://www.oedigital.com'+si['href'] for si in s1]
post_titles = [si.h2.text.strip() for si in s1]
post_dates = [extract_post_date(si) for si in s1]

In [198]:
any(post_date_i > datetime.strptime('2025', '%Y') for post_date_i in post_dates)

False

In [202]:
oe_posts = pd.DataFrame({'title': post_titles, 'link': post_links, 'date': post_dates})
oe_posts

,title,link,date
0,Brava Energia Hires Constellation’s Drilling R...,https://www.oedigital.com/news/519619-brava-en...,2024-11-26
1,Esgian Week 41 Report: Operators in Norway Pre...,https://www.oedigital.com/news/518013-esgian-w...,2024-10-13
2,Seadrill to Install Oil States’ MPD Tech on It...,https://www.oedigital.com/news/517173-seadrill...,2024-09-18
3,Archer Acquires Argentinian Managed Pressure D...,https://www.oedigital.com/news/515599-archer-a...,2024-08-01
4,Diamond Offshore Inks $89M Drillship Deal,https://www.oedigital.com/news/515316-diamond-...,2024-07-19
5,Diamond Offshore to Complete Upgrades on Its D...,https://www.oedigital.com/news/515247-diamond-...,2024-07-17
6,Diamond Offshore Nets $350M for Ocean BlackHaw...,https://www.oedigital.com/news/513759-diamond-...,2024-05-16
7,Seadrill Nets Two Drillship Commitments,https://www.oedigital.com/news/513530-seadrill...,2024-05-07
8,TotalEnergies Pays $199M for Tungsten Explorer...,https://www.oedigital.com/news/511363-totalene...,2024-02-07
9,Seadrill Secures $97.5M in New Drillship Contr...,https://www.oedigital.com/news/511150-seadrill...,2024-01-30
